# GPT-2 small (~124M) on TPU — JAX/Flax

Trains a 124M parameter GPT-2 small architecture on a ~1.2B-token slice of FineWeb-Edu.

**Where to actually run this (free TPU options, ranked):**

| platform | hardware | wall time for this run | how to access |
|---|---|---|---|
| **Kaggle** | TPU v3-8 (8 chips, ~1000 TFLOPS bf16) | **~2–4 h** | New notebook → Settings → Accelerator → "TPU VM v3-8". 30 hr/week free. |
| **TPU Research Cloud (TRC)** | v3-8 / v4-8 | ~1–4 h | Apply: https://sites.research.google/trc/about/ — usually approved in days. |
| Colab Pro+ | TPU v2-8 (sometimes) | ~3–6 h | $50/mo, TPU availability varies. |
| Colab Free | T4 GPU only — no TPU | ~24–48 h on T4 | Use the other notebook (`train_colab.ipynb`) for T4. |
| GCP direct (paid) | v5e-1 | ~10–20 h, ~$4–6 | `gcloud compute tpus tpu-vm create`. |

**Colab does NOT offer v5e-1 on any tier as of 2026-04.** If you want free TPU, Kaggle is the right answer.

This notebook auto-detects the accelerator and runs on TPU, GPU, or CPU.

## 1. Hardware check

In [ ]:
import os, subprocess
print('Detected hardware:')
if os.path.exists('/proc/cpuinfo'):
    print('  CPU available')
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode())
except Exception:
    print('  No NVIDIA GPU')
if os.environ.get('TPU_NAME') or os.path.exists('/dev/accel0') or os.environ.get('COLAB_TPU_ADDR'):
    print('  TPU detected')

## 2. Get the project files

Either `git clone` (option A) **or** upload the `tpu/` folder manually (option B).

In [ ]:
# Option A — clone (replace URL):
# !git clone https://github.com/<you>/gpt-2-nano.git
# %cd gpt-2-nano

# Option B — verify you uploaded the project root and are in it:
import os
assert os.path.isdir('tpu'), 'Run this from the gpt-2-nano project root (must contain a tpu/ folder)'
print('cwd:', os.getcwd())
print('tpu/ contents:', sorted(os.listdir('tpu')))

## 3. Install JAX (matched to your accelerator) + the rest

Pick the **one** install line for your hardware. The wrong wheel will silently run on CPU.

In [ ]:
# === TPU (Kaggle, Colab TPU runtime, GCP TPU VM) ===
!pip -q install --upgrade 'jax[tpu]' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

# === GPU (CUDA 12) ===
# !pip -q install --upgrade 'jax[cuda12]'

# === CPU only ===
# !pip -q install --upgrade jax

!pip -q install -r tpu/requirements.txt

In [ ]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Backend:', jax.default_backend())
print('Device count:', jax.device_count())

## 4. Mount Google Drive (optional but strongly recommended)

Colab/Kaggle sessions are wiped on disconnect. Storing checkpoints + tokenized data on Drive means:
* you don't re-tokenize 1.2B tokens every restart,
* you don't lose 3 hours of training to a disconnect.

Skip this cell if you're on a persistent VM (e.g. GCP TPU VM).

In [ ]:
# Colab:
from google.colab import drive
drive.mount('/content/drive')
PERSIST_DIR = '/content/drive/MyDrive/gpt-2-nano'

# Kaggle equivalent — use Kaggle Datasets (read-only) for input or /kaggle/working for output.
# PERSIST_DIR = '/kaggle/working/gpt-2-nano'

import os
os.makedirs(f'{PERSIST_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PERSIST_DIR}/data/fineweb_edu', exist_ok=True)
print('Persisting to:', PERSIST_DIR)

## 5. Sanity-check the model (1 forward pass)

In [ ]:
import jax, jax.numpy as jnp
from tpu.config import ModelConfig
from tpu.model import GPT, count_params

mcfg = ModelConfig()
model = GPT(mcfg, dtype=jnp.bfloat16)
rng = jax.random.PRNGKey(0)
x = jax.random.randint(rng, (2, 32), 0, mcfg.vocab_size)
params = model.init(rng, x, deterministic=True)['params']
print(f'parameters: {count_params(params) / 1e6:.2f}M')
logits = model.apply({'params': params}, x, deterministic=True)
print('logits shape:', logits.shape, '(expect (2, 32, 50304))')

## 6. Train

First run: tokenizes ~5 GB of FineWeb-Edu (~15–25 min on a Colab CPU), then starts training.
Reruns: skips tokenization, reloads the latest checkpoint from `--ckpt_dir`, resumes.

Checkpoints save **every 3 hours** of wall-clock time. Tune with `--ckpt_interval_hours`.

In [ ]:
# Use Drive paths if you ran cell 4; otherwise local paths (lost on disconnect).
DATA_DIR = f'{PERSIST_DIR}/data/fineweb_edu'
CKPT_DIR = f'{PERSIST_DIR}/checkpoints'

!python -m tpu.train \
    --data_dir "$DATA_DIR" \
    --ckpt_dir "$CKPT_DIR" \
    --ckpt_interval_hours 3.0 \
    --max_iters 6000

## 7. Sample from the trained model

In [ ]:
!python -m tpu.sample \
    --ckpt_dir "$CKPT_DIR" \
    --prompt "The history of mathematics begins" \
    --max_new_tokens 300 \
    --temperature 0.8 \
    --top_k 200

## 8. Resuming after a disconnect

If your Colab/Kaggle session times out, just rerun cells 1–3, then 6. The training loop:
1. Looks at `--ckpt_dir`.
2. Finds the latest saved step (restored from Google Drive).
3. Restores model + optimizer state and continues from that step.

Worst-case lost progress: ~3 hours (the checkpoint interval). Set `--ckpt_interval_hours 1.0` if you're getting frequent disconnects on a free tier.